In [1]:
!ls ../data/wiki*.jsonl

../data/wikipedia_synthetic10.jsonl  ../data/wikipedia_synthetic5.jsonl
../data/wikipedia_synthetic1.jsonl   ../data/wikipedia_synthetic6.jsonl
../data/wikipedia_synthetic2.jsonl   ../data/wikipedia_synthetic7.jsonl
../data/wikipedia_synthetic3.jsonl   ../data/wikipedia_synthetic8.jsonl
../data/wikipedia_synthetic4.jsonl   ../data/wikipedia_synthetic9.jsonl


In [2]:
import json
import pandas as pd
from glob import glob

PATH = "../data/wiki*.jsonl"
files = sorted(glob(PATH))
# files = [f for f in files if "8" in f or "9" in f ]

def load_json(file):
    def get_data(raw):
        line = json.loads(raw)
        return {
            "text": line["text"],
            "labels": line.get("labels"),
            "not_labels": line.get("not_labels")
        }
    with open(file, "r") as f:
        data = [get_data(line) for line in f]
    return data

files_data = [load_json(file) for file in files]
df = pd.DataFrame([i for file_data in files_data for i in file_data])

In [3]:
df.sample(5)

,text,labels,not_labels
34958,Welcome to the Aritzo region in Sardinia's mou...,"[reassuring tone, altitude awareness, visitor ...","[emergency contact information, symptom recogn..."
22994,The Rekem Museum's audio guide claimed I was s...,"[accuracy_complaint, refund_request, expectati...","[billing_dispute, repeated_contact_needed, ina..."
29319,"From a small town in Minnesota, their songs tr...","[journey_from_small_to_big, cultural_impact, c...","[friendship_and_collaboration, helping_others_..."
7595,"Good morning. I am Margaret Chen, a film histo...","[professional_expertise_assertion, uncertainty...","[temporal_specificity_claim, evidence_identifi..."
27171,Patient demonstrates significant confusion abo...,"[cultural_identity_confusion, fragmented_ident...","[historical_event_intrusive_thoughts, compulsi..."


In [4]:
def merge_group(group):
    merged_labels = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels  # remove any intersection
    return pd.Series({
        "labels": sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
    })

df = (
    df.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"{len(df)} unique texts after merging")
df.sample(5)


39794 unique texts after merging


,text,labels,not_labels
34870,Ovidiu Island floated in the middle of the mil...,"[bird_life_presence, gentle_natural_setting, h...","[anthropomorphic_nature, daily_rhythm, peacefu..."
39642,The bench must remain as principled as the bal...,"[constitutional_principles, declarative_values...","[electoral_mobilization, partisan_loyalty, tra..."
29307,Eternal Frames Studio announces its upcoming f...,"[animal_bride_motif_highlighted, award_recogni...","[afanasyev_attribution_included, historical_ma..."
1241,"In the early 1980s, the federal, provincial, a...",[multi-level government coordination mandate],"[arms-length development agency structure, mun..."
8138,5. Transition immediately from defensive recov...,"[attacking_contribution, recovery_movement, st...","[aerial_dominance, challenge_timing, leadershi..."


In [5]:
import random

random.seed(42)

# A label is present around 3x in the data, 
# so we adjust the test ratio to get ~5% test rows after the strict split 
# that also excludes rows with test labels in "not_labels"
test_ratio = 0.05 / 3 

# All labels (positive + negative)
all_pos_labels = set(l for labs in df["labels"] for l in labs)
all_neg_labels = set(l for labs in df["not_labels"] for l in labs)

full_vocab = all_pos_labels | all_neg_labels

# Only choose test labels from positive labels
# (because they need to be ground truth in test)
candidate_test_labels = list(all_pos_labels)
random.shuffle(candidate_test_labels)

n_test_labels = int(len(all_pos_labels) * test_ratio)
test_labels = set(candidate_test_labels[:n_test_labels])

# Strict split:
# Test rows = rows containing at least one test label
test_mask = df["labels"].apply(
    lambda labs: bool(set(labs) & test_labels)
)

df_test = df[test_mask].copy()

# Train rows = rows containing NO test label anywhere
train_mask = df.apply(
    lambda row: (
        len(set(row["labels"]) & test_labels) == 0
        and len(set(row["not_labels"]) & test_labels) == 0
    ),
    axis=1
)

df_train = df[train_mask].copy()

# Reset index
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Verify
train_seen = (
    set(l for labs in df_train["labels"] for l in labs)
    |
    set(l for labs in df_train["not_labels"] for l in labs)
)

test_seen = (
    set(l for labs in df_test["labels"] for l in labs)
    |
    set(l for labs in df_test["not_labels"] for l in labs)
)

intersection = test_labels & train_seen

print("Train rows:", len(df_train))
print("Test rows :", len(df_test))
print("Held-out test labels:", len(test_labels))
print("Intersection with train:", len(intersection))

Train rows: 35964
Test rows : 2134
Held-out test labels: 1451
Intersection with train: 0


In [6]:
import pandas as pd
from IPython.display import display

train_pos = set(lab for labs in df_train["labels"]     for lab in labs)
train_neg = set(lab for labs in df_train["not_labels"] for lab in labs)
test_pos  = set(lab for labs in df_test["labels"]      for lab in labs)
test_neg  = set(lab for labs in df_test["not_labels"]  for lab in labs)

total = len(test_pos) + len(test_neg)

# For a given target set, split by how the label was seen in train
def bucket(s):
    return {
        "seen in train as positive only":          len((s & train_pos) - train_neg),
        "seen in train as negative only":          len((s & train_neg) - train_pos),
        "seen in train as both pos and neg":       len(s & train_pos & train_neg),
        "never seen in train":                     len(s - train_pos - train_neg),
    }

tp = bucket(test_pos)   # labels used as ground-truth positives in test
tn = bucket(test_neg)   # labels used as hard negatives in test
index = list(tp.keys())

counts = pd.DataFrame({
    "used as positive in test": [tp[k] for k in index],
    "used as negative in test": [tn[k] for k in index],
    "total":                    [tp[k] + tn[k] for k in index],
}, index=index)
counts.index.name = "how the label was seen in train"

ratios = (counts / total * 100).round(1).astype(str) + "%"

print(f"Unique test positive labels: {len(test_pos)}   "
      f"Unique test negative labels: {len(test_neg)}   "
      f"Total: {total}\n")
print("── Counts ──")
display(counts)
print("── % of total test labels ──")
display(ratios)


Unique test positive labels: 6888   Unique test negative labels: 6605   Total: 13493

── Counts ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,812,1524,2336
seen in train as negative only,1493,1088,2581
seen in train as both pos and neg,1638,2681,4319
never seen in train,2945,1312,4257


── % of total test labels ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,6.0%,11.3%,17.3%
seen in train as negative only,11.1%,8.1%,19.1%
seen in train as both pos and neg,12.1%,19.9%,32.0%
never seen in train,21.8%,9.7%,31.5%


In [7]:
import datasets

train_ds = datasets.Dataset.from_pandas(df_train)
test_ds = datasets.Dataset.from_pandas(df_test)

dataset = datasets.DatasetDict({
    "train": train_ds,
    "test": test_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 35964
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 2134
    })
})

In [8]:
dataset.push_to_hub("alexneakameni/ZSHOT-HARDSET-v2", commit_description="Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2/commit/ac1043e30790c035ba97995a33761f25c31273ae', commit_message='Upload dataset', commit_description='Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.', oid='ac1043e30790c035ba97995a33761f25c31273ae', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-v2'), pr_revision=None, pr_num=None)